# CS224W Project Milestone — Dataset Processing
**Beyond Interactions: Knowledge Graph-Enhanced Product Recommendation**

This notebook builds deduplicated user–product interactions and typed product–metadata edges from Amazon Reviews 2023, then measures item sparsity and metadata connectivity.

Run cells in order in Colab. Electronics is the default. Add Movies & TV in the configuration cell to reproduce the comparison. The raw downloads, cache, and DuckDB tables require substantial disk space. The notebook uses the full selected categories for descriptive analysis.

In [ ]:
%pip -q install duckdb huggingface_hub pandas


## 1. Configure the run and download data

In [ ]:
import duckdb
import pandas as pd
from pathlib import Path
from huggingface_hub import hf_hub_download

REPO = "McAuley-Lab/Amazon-Reviews-2023"
CATEGORIES = ["Electronics"]  # Add "Movies_and_TV" for comparison.

WORKDIR = Path("/content/cs224w_milestone")
WORKDIR.mkdir(parents=True, exist_ok=True)

TMPDIR = WORKDIR / "tmp"
TMPDIR.mkdir(exist_ok=True)

con = duckdb.connect(str(WORKDIR / "amazon_eda.duckdb"))
con.execute("SET memory_limit='4GB'")
con.execute("SET threads=2")
con.execute(f"SET temp_directory='{TMPDIR}'")

# Metadata entities connected to 2–1000 reviewed products
MAX_ENTITY_DEGREE = 1000

print("DuckDB version:", duckdb.__version__)
print("Ready.")


In [ ]:
def get_paths(category):
    print(f"\nDownloading {category} reviews...")

    review_path = hf_hub_download(
        repo_id=REPO,
        filename=f"raw/review_categories/{category}.jsonl",
        repo_type="dataset"
    )

    print(f"Downloading {category} metadata...")

    metadata_path = hf_hub_download(
        repo_id=REPO,
        filename=f"raw/meta_categories/meta_{category}.jsonl",
        repo_type="dataset"
    )

    print(f"{category} ready.")
    return review_path, metadata_path


paths = {
    category: get_paths(category)
    for category in CATEGORIES
}


## 2. Build user–product interactions
Each distinct user–product pair counts as one interaction. These counts describe the full category.

In [ ]:
def build_interactions(category):
    suffix = category.lower()
    review_path = paths[category][0]

    con.execute(f"""
        CREATE OR REPLACE TABLE interactions_{suffix} AS
        SELECT DISTINCT
            user_id,
            parent_asin
        FROM read_json(
            ?,
            columns={{
                'user_id': 'VARCHAR',
                'parent_asin': 'VARCHAR'
            }},
            format='newline_delimited'
        )
        WHERE user_id IS NOT NULL
          AND parent_asin IS NOT NULL
    """, [review_path])

    con.execute(f"""
        CREATE OR REPLACE TABLE item_counts_{suffix} AS
        SELECT
            parent_asin,
            COUNT(*) AS n
        FROM interactions_{suffix}
        GROUP BY parent_asin
    """)

    return con.execute(f"""
        SELECT
            '{category}' AS category,
            COUNT(*) AS interactions,
            COUNT(DISTINCT user_id) AS users,
            COUNT(DISTINCT parent_asin) AS products
        FROM interactions_{suffix}
    """).df()


interaction_summary = pd.concat(
    [build_interactions(c) for c in CATEGORIES],
    ignore_index=True
)

display(interaction_summary)


In [ ]:
def sparsity_stats(category):
    suffix = category.lower()

    return con.execute(f"""
        SELECT
            '{category}' AS category,
            COUNT(*) AS products,
            ROUND(AVG(n), 2) AS avg_interactions,
            MEDIAN(n) AS median_interactions,
            ROUND(100.0 * AVG((n = 1)::INT), 2)
                AS pct_eq1,
            ROUND(100.0 * AVG((n <= 3)::INT), 2)
                AS pct_le3,
            ROUND(100.0 * AVG((n <= 5)::INT), 2)
                AS pct_le5,
            ROUND(100.0 * AVG((n <= 10)::INT), 2)
                AS pct_le10
        FROM item_counts_{suffix}
    """).df()


sparsity = pd.concat(
    [sparsity_stats(c) for c in CATEGORIES],
    ignore_index=True
)

display(sparsity)


## 3. Join metadata and build typed edges
Metadata rows are matched by parent ASIN. Entity IDs are scoped by relation, so a brand and a manufacturer with the same text are distinct entities.

In [ ]:
def build_metadata(category):
    suffix = category.lower()
    metadata_path = paths[category][1]

    con.execute(f"""
        CREATE OR REPLACE TABLE metadata_{suffix} AS
        WITH raw_metadata AS (
            SELECT
                parent_asin,
                store,
                main_category,
                categories,
                details
            FROM read_json(
                ?,
                format='newline_delimited',
                columns={{
                    'parent_asin': 'VARCHAR',
                    'store': 'VARCHAR',
                    'main_category': 'VARCHAR',
                    'categories': 'VARCHAR[]',
                    'details': 'JSON'
                }}
            )
            WHERE parent_asin IS NOT NULL
        ),
        dedup AS (
            SELECT *
            FROM raw_metadata
            QUALIFY ROW_NUMBER() OVER (
                PARTITION BY parent_asin
                ORDER BY CAST(details AS VARCHAR) NULLS LAST,
                         store, main_category, categories
            ) = 1
        )
        SELECT m.*
        FROM dedup m
        INNER JOIN item_counts_{suffix} i
            ON m.parent_asin = i.parent_asin
    """, [metadata_path])

    return con.execute(f"""
        SELECT
            '{category}' AS category,
            (SELECT COUNT(*) FROM item_counts_{suffix})
                AS reviewed_products,
            COUNT(*) AS metadata_matches,
            ROUND(
                100.0 * COUNT(*) /
                (SELECT COUNT(*) FROM item_counts_{suffix}),
                2
            ) AS metadata_coverage_pct
        FROM metadata_{suffix}
    """).df()


metadata_coverage = pd.concat(
    [build_metadata(c) for c in CATEGORIES],
    ignore_index=True
)

display(metadata_coverage)


In [ ]:
# Relations to keep
DOMAIN_RELATIONS = {
    "Movies_and_TV": {
        "studio": ["Studio"],
        "director": ["Director", "Directors"],
        "actor": ["Actors", "Actor", "Starring"],
        "producer": ["Producer", "Producers"]
    },
    "Electronics": {
        "brand": ["Brand"],
        "manufacturer": ["Manufacturer"],
        "model": ["Model Name", "Model Number"],
        "connectivity": ["Connectivity Technology"]
    }
}

# Only split fields that commonly contain multiple values.
MULTI_VALUE_RELATIONS = {"director", "actor", "producer"}

def build_domain_entities(category):
    suffix = category.lower()
    metadata_table = f"metadata_{suffix}"
    output_table = f"domain_entities_{suffix}"

    queries = []

    for relation, keys in DOMAIN_RELATIONS[category].items():
        for key in keys:
            json_path = '$."' + key + '"'

            # Extract JSON field as a string
            value_expr = (
                f"json_extract_string(details, '{json_path}')"
            )

            if relation in MULTI_VALUE_RELATIONS:
                # Split comma- and semicolon-separated values
                queries.append(f"""
                    SELECT
                        parent_asin,
                        '{relation}' AS relation,
                        LOWER(TRIM(value)) AS entity
                    FROM {metadata_table},
                         UNNEST(
                             regexp_split_to_array(
                                 COALESCE({value_expr}, ''),
                                 '[,;]'
                             )
                         ) AS t(value)
                """)
            else:
                queries.append(f"""
                    SELECT
                        parent_asin,
                        '{relation}' AS relation,
                        LOWER(TRIM({value_expr})) AS entity
                    FROM {metadata_table}
                """)

    union_sql = "\nUNION ALL\n".join(queries)

    con.execute(f"""
        CREATE OR REPLACE TABLE {output_table} AS
        SELECT DISTINCT
            parent_asin,
            relation,
            entity
        FROM ({union_sql})
        WHERE entity IS NOT NULL
          AND LENGTH(entity) >= 2
          AND entity NOT IN (
              'unknown', 'none', 'n/a', 'null', '[]'
          )
    """)

    return con.execute(f"""
        SELECT
            '{category}' AS category,
            relation,
            COUNT(*) AS edges,
            COUNT(DISTINCT entity) AS unique_entities
        FROM {output_table}
        GROUP BY relation
        ORDER BY relation
    """).df()


domain_relation_summary = pd.concat(
    [
        build_domain_entities(category)
        for category in CATEGORIES
    ],
    ignore_index=True
)

display(domain_relation_summary)


## 4. Measure metadata connectivity
An item has a shared metadata neighbor when at least one of its typed entities also links to another reviewed item. The capped metric counts only entities linked to 2–1,000 items.

In [ ]:
for category in CATEGORIES:
    suffix = category.lower()

    con.execute(f"""
        CREATE OR REPLACE TABLE domain_entity_degrees_{suffix} AS
        SELECT
            relation,
            entity,
            COUNT(DISTINCT parent_asin) AS degree
        FROM domain_entities_{suffix}
        GROUP BY relation, entity
    """)


In [ ]:
def domain_connectivity_stats(category):
    suffix = category.lower()

    con.execute(f"""
        CREATE OR REPLACE TABLE domain_product_flags_{suffix} AS
        WITH flags AS (
            SELECT
                e.parent_asin,

                MAX(
                    (d.degree >= 2)::INT
                ) AS has_shared,

                MAX(
                    (
                        d.degree BETWEEN 2
                        AND {MAX_ENTITY_DEGREE}
                    )::INT
                ) AS has_capped_shared

            FROM domain_entities_{suffix} e

            JOIN domain_entity_degrees_{suffix} d
              ON e.relation = d.relation
             AND e.entity = d.entity

            GROUP BY e.parent_asin
        )

        SELECT
            i.parent_asin,
            i.n,
            (f.parent_asin IS NOT NULL)::INT
                AS has_domain_metadata,
            COALESCE(f.has_shared, 0)
                AS has_shared,
            COALESCE(f.has_capped_shared, 0)
                AS has_capped_shared

        FROM item_counts_{suffix} i

        LEFT JOIN flags f
          ON i.parent_asin = f.parent_asin
    """)

    return con.execute(f"""
        SELECT
            '{category}' AS category,

            COUNT(*) AS products,

            ROUND(
                100.0 * AVG(has_domain_metadata),
                2
            ) AS domain_metadata_coverage_pct,

            ROUND(
                100.0 * AVG(
                    CASE WHEN n = 1
                    THEN has_domain_metadata END
                ), 2
            ) AS single_metadata_coverage_pct,

            ROUND(
                100.0 * AVG(
                    CASE WHEN n = 1
                    THEN has_shared END
                ), 2
            ) AS single_shared_pct,

            ROUND(
                100.0 * AVG(
                    CASE WHEN n = 1
                    THEN has_capped_shared END
                ), 2
            ) AS single_capped_pct,

            ROUND(
                100.0 * AVG(
                    CASE WHEN n <= 5
                    THEN has_capped_shared END
                ), 2
            ) AS le5_capped_pct

        FROM domain_product_flags_{suffix}
    """).df()


domain_comparison = pd.concat(
    [
        domain_connectivity_stats(category)
        for category in CATEGORIES
    ],
    ignore_index=True
)

display(domain_comparison.set_index("category").T)


In [ ]:
def domain_relation_connectivity(category):
    suffix = category.lower()

    return con.execute(f"""
        WITH relation_flags AS (
            SELECT
                e.parent_asin,
                e.relation,

                MAX(
                    (d.degree BETWEEN 2
                     AND {MAX_ENTITY_DEGREE})::INT
                ) AS connected

            FROM domain_entities_{suffix} e

            JOIN domain_entity_degrees_{suffix} d
              ON e.relation = d.relation
             AND e.entity = d.entity

            GROUP BY
                e.parent_asin,
                e.relation
        ),

        totals AS (
            SELECT COUNT(*) AS total_single
            FROM item_counts_{suffix}
            WHERE n = 1
        )

        SELECT
            '{category}' AS category,
            f.relation,

            COUNT_IF(i.n = 1)
                AS single_products_with_relation,

            COUNT_IF(
                i.n = 1 AND f.connected = 1
            ) AS connected_single_products,

            ROUND(
                100.0 * COUNT_IF(
                    i.n = 1 AND f.connected = 1
                ) / MAX(t.total_single),
                2
            ) AS total_single_connectivity_pct

        FROM relation_flags f

        JOIN item_counts_{suffix} i
          ON f.parent_asin = i.parent_asin

        CROSS JOIN totals t

        GROUP BY f.relation
        ORDER BY connected_single_products DESC
    """).df()


domain_relation_comparison = pd.concat(
    [
        domain_relation_connectivity(category)
        for category in CATEGORIES
    ],
    ignore_index=True
)

display(domain_relation_comparison)


## 5. Export milestone outputs
The CSVs summarize the analysis. The Parquet files contain the processed interaction pairs and metadata triples. These exports are for exploratory analysis. Temporal training splits will need review timestamps from the raw data.

In [ ]:
summary = (
    interaction_summary.merge(sparsity, on=["category", "products"])
    .merge(metadata_coverage, on="category")
    .merge(domain_comparison, on=["category", "products"])
)
display(summary.set_index("category").T)

summary.to_csv(WORKDIR / "category_summary.csv", index=False)
domain_relation_summary.to_csv(WORKDIR / "relation_summary.csv", index=False)
domain_relation_comparison.to_csv(WORKDIR / "relation_connectivity.csv", index=False)

for category in CATEGORIES:
    suffix = category.lower()
    for table in ("interactions", "domain_entities"):
        output_path = WORKDIR / f"{table}_{suffix}.parquet"
        con.table(f"{table}_{suffix}").write_parquet(str(output_path))

print("Outputs:", WORKDIR)


## Reference results from the original exploratory run

| Metric | Electronics | Movies & TV |
|---|---:|---:|
| Unique user–product pairs | 43,365,426 | 17,158,519 |
| Items with at most five interactions | 65.50% | 65.46% |
| Single-interaction items with a shared entity after degree filtering | 78.23% | 78.56% |

The last row measures graph connectivity. Recommendation performance is a separate experiment.